In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ==========================================================
# Step15.1.1
# Create Step15 directories
# ==========================================================

import os
import pandas as pd
import numpy as np

step15_dir = _hcc_path('LIHC_RNA_Step15_Machine_Learning')

input_dir = os.path.join(step15_dir, "Input")
results_dir = os.path.join(step15_dir, "Results")
figures_dir = os.path.join(step15_dir, "Figures")
models_dir = os.path.join(step15_dir, "Models")

for d in [
    step15_dir,
    input_dir,
    results_dir,
    figures_dir,
    models_dir
]:
    os.makedirs(
        d,
        exist_ok=True
    )

print("Step15 folders ready.")

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ==========================================================
# Load patient-level datasets
# ==========================================================

# 1. Expression
expression_all = pd.read_csv(
    _hcc_path('LIHC_RNA_Step11_Immune/Input/LIHC_Expression_365_log2CPM.tsv'),
    sep="\t",
    index_col=0
)

# 2. RiskScore + survival
risk_df = pd.read_csv(
    _hcc_path('LIHC_RNA_Step8_Final_Model/Results/Final_Top10_RiskScore.csv'),
    index_col=0
)

# 3. WGCNA module eigengenes
ME_matrix = pd.read_csv(
    _hcc_path('LIHC_RNA_Step13_Molecular_Subtyping/Input/LIHC_Module_Eigengenes_365.csv'),
    index_col=0
)

# 4. Molecular subtype solutions
Subtype_Solutions = pd.read_csv(
    _hcc_path('LIHC_RNA_Step13_Molecular_Subtyping/Results/LIHC_Retained_3_Molecular_Subtype_Solutions.csv'),
    index_col=0
)

# 5. Immune features
immune_df = pd.read_csv(
    _hcc_path('LIHC_RNA_Step11_Immune/Results/LIHC_Integrated_Immune_Matrix_365.csv'),
    index_col=0
)

# 6. Hallmark GSVA
GSVA_matrix = pd.read_csv(
    _hcc_path('LIHC_RNA_Step14_Pathway/Results/Hallmark_GSVA_Score_Matrix.csv'),
    index_col=0
)

# pathway × patient → patient × pathway
GSVA_patient = GSVA_matrix.T


print("Expression :", expression_all.shape)
print("Risk       :", risk_df.shape)
print("ME         :", ME_matrix.shape)
print("Subtype    :", Subtype_Solutions.shape)
print("Immune     :", immune_df.shape)
print("GSVA       :", GSVA_patient.shape)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ==========================================================
# Clinical data
# ==========================================================

clinical_df = pd.read_csv(
    _hcc_path('LIHC_clinical_clean.csv')
)

clinical_tumor = (
    clinical_df[
        clinical_df["Tumor_Normal"] == "Tumor"
    ]
    .drop_duplicates(
        subset="Sample_ID"
    )
    .set_index("Sample_ID")
)

clinical_features = clinical_tumor[
    [
        "Age",
        "Gender",
        "Stage",
        "T",
        "N",
        "M"
    ]
].copy()

print(
    "Clinical:",
    clinical_features.shape
)

In [ ]:
# ==========================================================
# Gene-level features
# ==========================================================

prognostic_genes_10 = [
    "HOXD9",
    "RFESD",
    "RPSAP53",
    "TRGC1",
    "FAM227B",
    "AC015871.6",
    "LINC00324",
    "ACOXL",
    "PTMAP4",
    "ANKRD20A7P"
]


Blue_genes = [
    "CENPA",
    "KIF2C",
    "CCNB1",
    "CDC20",
    "PRR11",
    "CDCA8"
]

Red_genes = [
    "SLC52A2",
    "RPL8",
    "BOP1",
    "ZFAS1",
    "SNHG6",
    "RPL7P9"
]

Brown_genes = [
    "CYP8B1",
    "GLYATL1",
    "PLG",
    "ACSM2A",
    "ABAT",
    "MAT1A"
]


Hub_genes_18 = (
    Blue_genes
    +
    Red_genes
    +
    Brown_genes
)


gene_features_requested = list(
    dict.fromkeys(
        prognostic_genes_10
        +
        Hub_genes_18
    )
)


genes_available = [
    g for g in gene_features_requested
    if g in expression_all.index
]

genes_missing = [
    g for g in gene_features_requested
    if g not in expression_all.index
]


print(
    "Requested genes:",
    len(gene_features_requested)
)

print(
    "Available genes:",
    len(genes_available)
)

print(
    "Missing genes:",
    genes_missing
)


gene_features = (
    expression_all
    .loc[
        genes_available
    ]
    .T
)

print(
    "Gene feature matrix:",
    gene_features.shape
)

In [ ]:
# ==========================================================
# Build Master Feature Table
# ==========================================================

master_df = (
    risk_df[
        [
            "OS_time",
            "OS",
            "RiskScore"
        ]
    ]
    .join(
        clinical_features,
        how="left"
    )
    .join(
        gene_features,
        how="left"
    )
    .join(
        ME_matrix[
            [
                "ME_blue",
                "ME_red",
                "ME_brown"
            ]
        ],
        how="left"
    )
    .join(
        Subtype_Solutions,
        how="left"
    )
    .join(
        immune_df,
        how="left"
    )
    .join(
        GSVA_patient,
        how="left",
        rsuffix="_GSVA"
    )
)


print(
    "Master table shape:",
    master_df.shape
)

master_df.head()

In [ ]:
# ==========================================================
# Master table QC
# ==========================================================

print(
    "Patients:",
    master_df.shape[0]
)

print(
    "Features/columns:",
    master_df.shape[1]
)


print("\nOS events:")
print(
    master_df["OS"]
    .value_counts(
        dropna=False
    )
)


print("\nMissing values TOP20:")

display(
    master_df
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(20)
)

In [ ]:
master_df.to_csv(
    os.path.join(
        input_dir,
        "LIHC_Multilevel_Master_Feature_Table.csv"
    )
)

print(
    "Master feature table saved."
)

In [ ]:
# ==========================================================
# Step15.2.1
# Define feature groups
# ==========================================================

# ----------------------------------------------------------
# Outcome variables
# 绝对不能作为预测特征
# ----------------------------------------------------------

outcome_cols = [
    "OS_time",
    "OS"
]


# ----------------------------------------------------------
# Clinical features
# ----------------------------------------------------------

clinical_cols = [
    "Age",
    "Gender",
    "Stage",
    "T",
    "N",
    "M"
]


# ----------------------------------------------------------
# 10-gene prognostic signature
# ----------------------------------------------------------

prognostic_gene_cols = [
    "HOXD9",
    "RFESD",
    "RPSAP53",
    "TRGC1",
    "FAM227B",
    "AC015871.6",
    "LINC00324",
    "ACOXL",
    "PTMAP4",
    "ANKRD20A7P"
]


# ----------------------------------------------------------
# 18 WGCNA Hub genes
# ----------------------------------------------------------

hub_gene_cols = [
    "CENPA",
    "KIF2C",
    "CCNB1",
    "CDC20",
    "PRR11",
    "CDCA8",

    "SLC52A2",
    "RPL8",
    "BOP1",
    "ZFAS1",
    "SNHG6",
    "RPL7P9",

    "CYP8B1",
    "GLYATL1",
    "PLG",
    "ACSM2A",
    "ABAT",
    "MAT1A"
]


# ----------------------------------------------------------
# Risk signature
# ----------------------------------------------------------

risk_cols = [
    "RiskScore"
]


# ----------------------------------------------------------
# WGCNA module-level features
# ----------------------------------------------------------

wgcna_cols = [
    "ME_blue",
    "ME_red",
    "ME_brown"
]


# ----------------------------------------------------------
# Molecular subtype features
# 暂时全部保留
# ----------------------------------------------------------

subtype_cols = [
    col for col in master_df.columns
    if (
        "Hub18_K2" in col
        or
        "Top30_each_K2" in col
        or
        "MM80_GS20_K2" in col
    )
]


# ----------------------------------------------------------
# Immune features
# 从immune_df自动获取
# ----------------------------------------------------------

immune_cols = [
    col for col in immune_df.columns
    if col in master_df.columns
]


# ----------------------------------------------------------
# Hallmark pathway features
# ----------------------------------------------------------

gsva_cols = [
    col for col in GSVA_patient.columns
    if col in master_df.columns
]


print("Outcome:", len(outcome_cols))
print("Clinical:", len(clinical_cols))
print("Prognostic genes:", len(prognostic_gene_cols))
print("Hub genes:", len(hub_gene_cols))
print("Risk:", len(risk_cols))
print("WGCNA:", len(wgcna_cols))
print("Subtype:", len(subtype_cols))
print("Immune:", len(immune_cols))
print("GSVA:", len(gsva_cols))

In [ ]:
subtype_numeric_cols = [
    "A_Hub18_K2",
    "D_Top30_each_K2",
    "E_MM80_GS20_K2"
]

print(
    master_df[
        subtype_numeric_cols
    ].head()
)

In [ ]:
# ==========================================================
# Agreement among retained subtype solutions
# ==========================================================

for col1 in subtype_numeric_cols:

    for col2 in subtype_numeric_cols:

        if col1 < col2:

            agreement = (
                master_df[col1]
                ==
                master_df[col2]
            ).mean()

            print(
                col1,
                "vs",
                col2,
                ":",
                round(
                    agreement,
                    4
                )
            )

In [ ]:
# ==========================================================
# Step15.2.4
# Clean clinical variables
# ==========================================================

ml_df = master_df.copy()


# ----------------------------------------------------------
# Gender
# Female = 0
# Male   = 1
# ----------------------------------------------------------

ml_df["Gender_binary"] = (
    ml_df["Gender"]
    .astype(str)
    .str.lower()
    .map({
        "female": 0,
        "male": 1
    })
)


# ----------------------------------------------------------
# Stage
# I = 1
# II = 2
# III = 3
# IV = 4
# ----------------------------------------------------------

def encode_stage(x):

    if pd.isna(x):
        return np.nan

    x = str(x).upper().strip()

    if x.startswith("STAGE IV"):
        return 4

    if x.startswith("STAGE III"):
        return 3

    if x.startswith("STAGE II"):
        return 2

    if x.startswith("STAGE I"):
        return 1

    return np.nan


ml_df["Stage_numeric"] = (
    ml_df["Stage"]
    .apply(
        encode_stage
    )
)


# ----------------------------------------------------------
# T stage
# T1/T1A/T1B -> 1
# T2          -> 2
# T3...       -> 3
# T4          -> 4
# TX          -> missing
# ----------------------------------------------------------

def encode_T(x):

    if pd.isna(x):
        return np.nan

    x = str(x).upper().strip()

    if x.startswith("T1"):
        return 1

    if x.startswith("T2"):
        return 2

    if x.startswith("T3"):
        return 3

    if x.startswith("T4"):
        return 4

    return np.nan


ml_df["T_numeric"] = (
    ml_df["T"]
    .apply(
        encode_T
    )
)


# ----------------------------------------------------------
# N stage
# N0 = 0
# N1 = 1
# NX = missing
# ----------------------------------------------------------

def encode_N(x):

    if pd.isna(x):
        return np.nan

    x = str(x).upper().strip()

    if x == "N0":
        return 0

    if x.startswith("N1"):
        return 1

    return np.nan


ml_df["N_numeric"] = (
    ml_df["N"]
    .apply(
        encode_N
    )
)


# ----------------------------------------------------------
# M stage
# M0 = 0
# M1 = 1
# MX = missing
# ----------------------------------------------------------

def encode_M(x):

    if pd.isna(x):
        return np.nan

    x = str(x).upper().strip()

    if x == "M0":
        return 0

    if x.startswith("M1"):
        return 1

    return np.nan


ml_df["M_numeric"] = (
    ml_df["M"]
    .apply(
        encode_M
    )
)

In [ ]:
clinical_numeric_cols = [
    "Age",
    "Gender_binary",
    "Stage_numeric",
    "T_numeric",
    "N_numeric",
    "M_numeric"
]


print(
    ml_df[
        clinical_numeric_cols
    ]
    .isna()
    .sum()
)

print("\nClinical numeric distributions:")

for col in clinical_numeric_cols:

    print("\n", col)

    print(
        ml_df[col]
        .value_counts(
            dropna=False
        )
        .sort_index()
    )

In [ ]:
# ==========================================================
# Feature families
# ==========================================================

feature_families = {

    "Clinical":
        clinical_numeric_cols,

    "PrognosticGenes10":
        prognostic_gene_cols,

    "HubGenes18":
        hub_gene_cols,

    "RiskScore":
        risk_cols,

    "WGCNA":
        wgcna_cols,

    "Subtype":
        subtype_numeric_cols,

    "Immune":
        immune_cols,

    "Hallmark_GSVA":
        gsva_cols
}


feature_family_summary = []

for family, cols in feature_families.items():

    available = [
        c for c in cols
        if c in ml_df.columns
    ]

    feature_family_summary.append({

        "Feature_family":
            family,

        "N_features":
            len(available),

        "Features":
            "; ".join(
                available
            )
    })


feature_family_summary = pd.DataFrame(
    feature_family_summary
)

display(
    feature_family_summary
)

In [ ]:
feature_family_summary.to_csv(
    os.path.join(
        input_dir,
        "Step15_Feature_Family_Summary.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step15.3.1
# Define survival modeling cohort
# ==========================================================

model_base_cols = [
    "OS_time",
    "OS",
    "Age",
    "Gender_binary",
    "Stage_numeric",
    "T_numeric"
]


model_cohort = (
    ml_df[
        model_base_cols
    ]
    .dropna()
    .copy()
)


print(
    "Modeling cohort:",
    model_cohort.shape
)

print(
    "Events:",
    int(
        model_cohort["OS"].sum()
    )
)

print(
    "Event rate:",
    model_cohort["OS"].mean()
)

In [ ]:
# ==========================================================
# Step15.3.2
# Train / Test split
# 70% / 30%
# ==========================================================

from sklearn.model_selection import train_test_split


model_samples = (
    model_cohort.index
)


train_samples, test_samples = train_test_split(
    model_samples,

    test_size=0.30,

    random_state=42,

    stratify=model_cohort.loc[
        model_samples,
        "OS"
    ]
)


print(
    "Train:",
    len(train_samples)
)

print(
    "Test:",
    len(test_samples)
)


print(
    "\nTrain events:",
    int(
        ml_df.loc[
            train_samples,
            "OS"
        ].sum()
    )
)

print(
    "Test events:",
    int(
        ml_df.loc[
            test_samples,
            "OS"
        ].sum()
    )
)

In [ ]:
# ==========================================================
# Save train/test split
# ==========================================================

split_df = pd.DataFrame(
    index=model_samples
)

split_df["Dataset"] = np.where(
    split_df.index.isin(
        train_samples
    ),
    "Train",
    "Test"
)


split_df["OS"] = (
    ml_df.loc[
        split_df.index,
        "OS"
    ]
)

split_df["OS_time"] = (
    ml_df.loc[
        split_df.index,
        "OS_time"
    ]
)


split_df.to_csv(
    os.path.join(
        input_dir,
        "LIHC_Survival_Train_Test_Split_70_30.csv"
    )
)


print(
    split_df["Dataset"]
    .value_counts()
)

In [ ]:
clinical_model_features = [
    "Age",
    "Gender_binary",
    "Stage_numeric",
    "T_numeric"
]

In [ ]:
gene_model_features = list(
    dict.fromkeys(
        prognostic_gene_cols
        +
        hub_gene_cols
    )
)

print(
    "Gene features:",
    len(gene_model_features)
)

In [ ]:
biological_model_features = list(
    dict.fromkeys(
        wgcna_cols
        +
        subtype_numeric_cols
        +
        immune_cols
        +
        gsva_cols
    )
)

print(
    "Biological features:",
    len(biological_model_features)
)

In [ ]:
integrated_feature_pool = list(
    dict.fromkeys(
        clinical_model_features
        +
        gene_model_features
        +
        biological_model_features
    )
)

print(
    "Integrated candidate features:",
    len(
        integrated_feature_pool
    )
)

In [ ]:
benchmark_feature_sets = {

    "Clinical":
        clinical_model_features,

    "RiskScore":
        [
            "RiskScore"
        ],

    "Clinical_RiskScore":
        clinical_model_features
        +
        [
            "RiskScore"
        ],

    "Gene":
        gene_model_features,

    "Biological":
        biological_model_features,

    "Integrated":
        integrated_feature_pool
}

In [ ]:
for name, cols in benchmark_feature_sets.items():

    print(
        name,
        ":",
        len(cols),
        "features"
    )

In [ ]:
# ==========================================================
# Step15.4.1
# Prepare train/test survival datasets
# ==========================================================

train_df = ml_df.loc[
    train_samples
].copy()

test_df = ml_df.loc[
    test_samples
].copy()


print("Train:", train_df.shape)
print("Test :", test_df.shape)

print("\nTrain events:")
print(train_df["OS"].value_counts())

print("\nTest events:")
print(test_df["OS"].value_counts())

In [ ]:
# ==========================================================
# Step15.4.2
# Univariate Cox screening
# Gene features
# TRAINING SET ONLY
# ==========================================================

from lifelines import CoxPHFitter
import pandas as pd
import numpy as np


uni_cox_results = []


for gene in gene_model_features:

    temp = train_df[
        [
            "OS_time",
            "OS",
            gene
        ]
    ].dropna()

    try:

        cph = CoxPHFitter()

        cph.fit(
            temp,
            duration_col="OS_time",
            event_col="OS"
        )

        row = cph.summary.loc[gene]

        uni_cox_results.append({

            "Gene": gene,

            "HR":
                row["exp(coef)"],

            "HR_lower95":
                row["exp(coef) lower 95%"],

            "HR_upper95":
                row["exp(coef) upper 95%"],

            "P_value":
                row["p"]

        })

    except Exception as e:

        print(
            "Failed:",
            gene,
            e
        )


Gene_UniCox = pd.DataFrame(
    uni_cox_results
)

In [ ]:
from statsmodels.stats.multitest import multipletests


Gene_UniCox["FDR"] = multipletests(
    Gene_UniCox["P_value"],
    method="fdr_bh"
)[1]


Gene_UniCox = (
    Gene_UniCox
    .sort_values(
        "P_value"
    )
    .reset_index(
        drop=True
    )
)


display(
    Gene_UniCox
)

In [ ]:
Gene_UniCox.to_csv(
    os.path.join(
        results_dir,
        "Gene_Training_Univariate_Cox.csv"
    ),
    index=False
)

In [ ]:
Gene_UniCox["Direction"] = np.where(
    Gene_UniCox["HR"] > 1,
    "Risk",
    "Protective"
)

display(
    Gene_UniCox[
        [
            "Gene",
            "HR",
            "HR_lower95",
            "HR_upper95",
            "P_value",
            "FDR",
            "Direction"
        ]
    ]
)

In [ ]:
gene_candidates_uni = (
    Gene_UniCox.loc[
        Gene_UniCox["P_value"] < 0.05,
        "Gene"
    ]
    .tolist()
)


print(
    "Univariate Cox candidate genes:",
    len(gene_candidates_uni)
)

print(
    gene_candidates_uni
)

In [ ]:
gene_candidates_fdr = (
    Gene_UniCox.loc[
        Gene_UniCox["FDR"] < 0.05,
        "Gene"
    ]
    .tolist()
)


print(
    "\nFDR-significant genes:",
    len(gene_candidates_fdr)
)

print(
    gene_candidates_fdr
)

In [ ]:
# ==========================================================
# Correlation among Cox candidate genes
# ==========================================================

gene_corr = (
    train_df[
        gene_candidates_uni
    ]
    .corr(
        method="spearman"
    )
)


print(
    "Correlation matrix:",
    gene_corr.shape
)

In [ ]:
high_corr_pairs = []

for i in range(len(gene_corr.columns)):

    for j in range(i + 1, len(gene_corr.columns)):

        g1 = gene_corr.columns[i]
        g2 = gene_corr.columns[j]

        rho = gene_corr.iloc[i, j]

        if abs(rho) >= 0.80:

            high_corr_pairs.append({
                "Gene1": g1,
                "Gene2": g2,
                "Spearman_rho": rho
            })


High_Corr_Gene_Pairs = pd.DataFrame(
    high_corr_pairs
)


print(
    "Highly correlated pairs |rho| >= 0.80:",
    len(High_Corr_Gene_Pairs)
)

display(
    High_Corr_Gene_Pairs
)

In [ ]:
High_Corr_Gene_Pairs.to_csv(
    os.path.join(
        results_dir,
        "Gene_Training_High_Correlation_Pairs.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Prepare gene matrix for Elastic Net Cox
# TRAINING SET ONLY
# ==========================================================

from sklearn.preprocessing import StandardScaler
from sksurv.util import Surv

elastic_genes = gene_candidates_uni.copy()

X_train_gene = (
    train_df[
        elastic_genes
    ]
    .copy()
)

y_train_surv = Surv.from_arrays(
    event=train_df["OS"].astype(bool),
    time=train_df["OS_time"].astype(float)
)


print("X train:", X_train_gene.shape)
print("Genes:", elastic_genes)

In [ ]:
# ==========================================================
# Standardization
# ==========================================================

scaler_gene = StandardScaler()

X_train_gene_scaled = scaler_gene.fit_transform(
    X_train_gene
)

X_train_gene_scaled = pd.DataFrame(
    X_train_gene_scaled,
    index=X_train_gene.index,
    columns=X_train_gene.columns
)

print(X_train_gene_scaled.shape)

In [ ]:
# ==========================================================
# Elastic Net Cox
# Cross-validation for alpha
# ==========================================================

import numpy as np

from sksurv.linear_model import CoxnetSurvivalAnalysis
from sklearn.model_selection import KFold
from sksurv.metrics import concordance_index_censored


# ----------------------------------------------------------
# First fit once to obtain alpha path
# ----------------------------------------------------------

coxnet_path = CoxnetSurvivalAnalysis(
    l1_ratio=0.5,
    alpha_min_ratio=0.01,
    n_alphas=100,
    max_iter=100000
)

coxnet_path.fit(
    X_train_gene_scaled,
    y_train_surv
)


alphas = coxnet_path.alphas_

print("Number of alphas:", len(alphas))
print("Alpha range:", alphas.min(), "-", alphas.max())

In [ ]:
# ==========================================================
# 5-fold CV
# ==========================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


alpha_cv_results = []


for alpha in alphas:

    fold_cindices = []

    for train_idx, val_idx in kf.split(
        X_train_gene_scaled
    ):

        X_tr = X_train_gene_scaled.iloc[
            train_idx
        ]

        X_val = X_train_gene_scaled.iloc[
            val_idx
        ]

        y_tr = y_train_surv[
            train_idx
        ]

        y_val = y_train_surv[
            val_idx
        ]


        model = CoxnetSurvivalAnalysis(
            l1_ratio=0.5,
            alphas=[alpha],
            max_iter=100000
        )

        try:

            model.fit(
                X_tr,
                y_tr
            )

            risk_pred = model.predict(
                X_val
            )

            c_index = concordance_index_censored(
                y_val["event"],
                y_val["time"],
                risk_pred
            )[0]

            fold_cindices.append(
                c_index
            )

        except Exception:
            continue


    if len(fold_cindices) > 0:

        alpha_cv_results.append({

            "Alpha":
                alpha,

            "Mean_Cindex":
                np.mean(
                    fold_cindices
                ),

            "SD_Cindex":
                np.std(
                    fold_cindices
                )

        })


ElasticNet_CV = pd.DataFrame(
    alpha_cv_results
)

ElasticNet_CV = (
    ElasticNet_CV
    .sort_values(
        "Mean_Cindex",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


ElasticNet_CV.head(10)

In [ ]:
best_alpha = (
    ElasticNet_CV
    .iloc[0]["Alpha"]
)

best_cv_cindex = (
    ElasticNet_CV
    .iloc[0]["Mean_Cindex"]
)


print(
    "Best alpha:",
    best_alpha
)

print(
    "Best CV C-index:",
    best_cv_cindex
)

In [ ]:
# ==========================================================
# Final Elastic Net Cox on training set
# ==========================================================

elastic_model = CoxnetSurvivalAnalysis(
    l1_ratio=0.5,
    alphas=[best_alpha],
    max_iter=100000
)

elastic_model.fit(
    X_train_gene_scaled,
    y_train_surv
)

In [ ]:
coef = (
    elastic_model
    .coef_
    .ravel()
)


ElasticNet_Coefficients = pd.DataFrame({

    "Gene":
        elastic_genes,

    "Coefficient":
        coef

})


ElasticNet_Coefficients[
    "Abs_Coefficient"
] = (
    ElasticNet_Coefficients[
        "Coefficient"
    ].abs()
)


ElasticNet_Coefficients[
    "Selected"
] = (
    ElasticNet_Coefficients[
        "Abs_Coefficient"
    ] > 1e-8
)


ElasticNet_Coefficients = (
    ElasticNet_Coefficients
    .sort_values(
        "Abs_Coefficient",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    ElasticNet_Coefficients
)

In [ ]:
elastic_selected_genes = (
    ElasticNet_Coefficients.loc[
        ElasticNet_Coefficients[
            "Selected"
        ],
        "Gene"
    ]
    .tolist()
)


print(
    "Elastic Net selected genes:",
    len(
        elastic_selected_genes
    )
)

print(
    elastic_selected_genes
)

In [ ]:
ElasticNet_CV.to_csv(
    os.path.join(
        results_dir,
        "Gene_ElasticNet_CV.csv"
    ),
    index=False
)


ElasticNet_Coefficients.to_csv(
    os.path.join(
        results_dir,
        "Gene_ElasticNet_Coefficients.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step15.6.1
# Random Survival Forest
# TRAINING SET ONLY
# ==========================================================

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

X_train_rsf = train_df[
    gene_candidates_uni
].copy()

y_train_rsf = Surv.from_arrays(
    event=train_df["OS"].astype(bool),
    time=train_df["OS_time"].astype(float)
)

print("RSF X:", X_train_rsf.shape)
print("RSF genes:", len(gene_candidates_uni))

In [ ]:
# ==========================================================
# Step15.6.2
# Fit Random Survival Forest
# ==========================================================

rsf_model = RandomSurvivalForest(
    n_estimators=1000,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rsf_model.fit(
    X_train_rsf,
    y_train_rsf
)

train_rsf_cindex = rsf_model.score(
    X_train_rsf,
    y_train_rsf
)

print(
    "RSF training C-index:",
    train_rsf_cindex
)

In [ ]:
# ==========================================================
# Step15.6.3
# Permutation importance
# ==========================================================

from sklearn.inspection import permutation_importance

rsf_perm = permutation_importance(
    rsf_model,
    X_train_rsf,
    y_train_rsf,

    n_repeats=50,
    random_state=42,
    n_jobs=-1
)

In [ ]:
RSF_Importance = pd.DataFrame({

    "Gene":
        gene_candidates_uni,

    "Importance_mean":
        rsf_perm.importances_mean,

    "Importance_SD":
        rsf_perm.importances_std

})

RSF_Importance = (
    RSF_Importance
    .sort_values(
        "Importance_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    RSF_Importance
)

In [ ]:
RSF_Importance.to_csv(
    os.path.join(
        results_dir,
        "Gene_RSF_Permutation_Importance.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step15.6.5
# Integrate Uni-Cox + Elastic Net + RSF
# ==========================================================

Gene_Selection_Integrated = (
    Gene_UniCox[
        [
            "Gene",
            "HR",
            "P_value",
            "FDR",
            "Direction"
        ]
    ]
    .merge(
        ElasticNet_Coefficients[
            [
                "Gene",
                "Coefficient",
                "Abs_Coefficient",
                "Selected"
            ]
        ],
        on="Gene",
        how="left"
    )
    .merge(
        RSF_Importance[
            [
                "Gene",
                "Importance_mean",
                "Importance_SD"
            ]
        ],
        on="Gene",
        how="left"
    )
)

In [ ]:
rsf_top10_genes = (
    RSF_Importance
    .head(10)["Gene"]
    .tolist()
)

print(rsf_top10_genes)

In [ ]:
Gene_Selection_Integrated["UniCox_significant"] = (
    Gene_Selection_Integrated["P_value"] < 0.05
)

Gene_Selection_Integrated["ElasticNet_selected"] = (
    Gene_Selection_Integrated["Selected"]
    .fillna(False)
)

Gene_Selection_Integrated["RSF_Top10"] = (
    Gene_Selection_Integrated["Gene"]
    .isin(rsf_top10_genes)
)


Gene_Selection_Integrated["Selection_Count"] = (
    Gene_Selection_Integrated[
        "UniCox_significant"
    ].astype(int)
    +
    Gene_Selection_Integrated[
        "ElasticNet_selected"
    ].astype(int)
    +
    Gene_Selection_Integrated[
        "RSF_Top10"
    ].astype(int)
)

In [ ]:
Gene_Selection_Integrated = (
    Gene_Selection_Integrated
    .sort_values(
        [
            "Selection_Count",
            "Importance_mean",
            "Abs_Coefficient"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)


display(
    Gene_Selection_Integrated[
        [
            "Gene",
            "HR",
            "Direction",
            "FDR",
            "Abs_Coefficient",
            "Importance_mean",
            "UniCox_significant",
            "ElasticNet_selected",
            "RSF_Top10",
            "Selection_Count"
        ]
    ]
)

In [ ]:
Gene_Selection_Integrated.to_csv(
    os.path.join(
        results_dir,
        "Gene_MultiMethod_Selection_Integrated.csv"
    ),
    index=False
)

In [ ]:
stable_genes = (
    Gene_Selection_Integrated.loc[
        Gene_Selection_Integrated[
            "Selection_Count"
        ] == 3,
        "Gene"
    ]
    .tolist()
)


print(
    "Stable genes selected by all 3 methods:",
    len(stable_genes)
)

print(
    stable_genes
)

In [ ]:
# ==========================================================
# Step15.7.1
# Stable prognostic genes selected by all 3 methods
# ==========================================================

stable_genes = [
    "HOXD9",
    "RPSAP53",
    "RFESD",
    "LINC00324",
    "TRGC1",
    "ANKRD20A7P",
    "FAM227B",
    "BOP1",
    "GLYATL1",
    "PRR11"
]

print("Stable genes:", len(stable_genes))
print(stable_genes)

In [ ]:
pd.DataFrame({
    "Gene": stable_genes
}).to_csv(
    os.path.join(
        results_dir,
        "Stable_Prognostic_Genes_10.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step15.7.1
# Define core feature sets
# ==========================================================

core_feature_sets = {

    "Clinical": [
        "Age",
        "Gender_binary",
        "Stage_numeric",
        "T_numeric"
    ],

    "RiskScore": [
        "RiskScore"
    ],

    "Clinical_RiskScore": [
        "Age",
        "Gender_binary",
        "Stage_numeric",
        "T_numeric",
        "RiskScore"
    ],

    "StableGenes": stable_genes,

    "Clinical_StableGenes": [
        "Age",
        "Gender_binary",
        "Stage_numeric",
        "T_numeric"
    ] + stable_genes
}


for name, features in core_feature_sets.items():
    print(name, len(features))

In [ ]:
# ==========================================================
# Step15.7.2
# CoxPH benchmark
# ==========================================================

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
import pandas as pd

cox_benchmark_results = []


for model_name, features in core_feature_sets.items():

    train_model = (
        train_df[
            ["OS_time", "OS"] + features
        ]
        .dropna()
        .copy()
    )

    test_model = (
        test_df[
            ["OS_time", "OS"] + features
        ]
        .dropna()
        .copy()
    )


    cph = CoxPHFitter(
        penalizer=0.01
    )

    cph.fit(
        train_model,
        duration_col="OS_time",
        event_col="OS"
    )


    # higher partial hazard = higher risk
    train_risk = (
        cph.predict_partial_hazard(
            train_model[features]
        )
        .values
        .ravel()
    )

    test_risk = (
        cph.predict_partial_hazard(
            test_model[features]
        )
        .values
        .ravel()
    )


    train_cindex = concordance_index(
        train_model["OS_time"],
        -train_risk,
        train_model["OS"]
    )

    test_cindex = concordance_index(
        test_model["OS_time"],
        -test_risk,
        test_model["OS"]
    )


    cox_benchmark_results.append({

        "Model": model_name,

        "N_features": len(features),

        "Train_n": len(train_model),

        "Test_n": len(test_model),

        "Train_Cindex": train_cindex,

        "Test_Cindex": test_cindex,

        "Overfit_gap":
            train_cindex - test_cindex
    })


Cox_Benchmark = pd.DataFrame(
    cox_benchmark_results
)

display(Cox_Benchmark)

In [ ]:
# ==========================================================
# Save benchmark
# ==========================================================

Cox_Benchmark.to_csv(
    os.path.join(
        results_dir,
        "Core_Feature_Sets_CoxPH_Benchmark.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step15.8.1
# Biological feature pool
# ==========================================================

biological_feature_groups = {}

for f in wgcna_cols:
    biological_feature_groups[f] = "WGCNA"

for f in subtype_numeric_cols:
    biological_feature_groups[f] = "Subtype"

for f in immune_cols:
    biological_feature_groups[f] = "Immune"

for f in gsva_cols:
    biological_feature_groups[f] = "Hallmark_GSVA"


biological_features = list(
    biological_feature_groups.keys()
)


print(
    "Total biological features:",
    len(biological_features)
)

print(pd.Series(
    biological_feature_groups
).value_counts())

In [ ]:
# ==========================================================
# Step15.8.2
# Univariate Cox screening
# Biological features
# TRAIN SET ONLY
# ==========================================================

from lifelines import CoxPHFitter
import pandas as pd
import numpy as np


bio_uni_results = []


for feature in biological_features:

    temp = train_df[
        [
            "OS_time",
            "OS",
            feature
        ]
    ].dropna().copy()


    # 必须有变异
    if temp[feature].nunique() <= 1:
        continue


    try:

        cph = CoxPHFitter()

        cph.fit(
            temp,
            duration_col="OS_time",
            event_col="OS"
        )


        row = cph.summary.loc[feature]


        bio_uni_results.append({

            "Feature":
                feature,

            "Feature_family":
                biological_feature_groups[
                    feature
                ],

            "N":
                len(temp),

            "HR":
                row["exp(coef)"],

            "HR_lower95":
                row[
                    "exp(coef) lower 95%"
                ],

            "HR_upper95":
                row[
                    "exp(coef) upper 95%"
                ],

            "P_value":
                row["p"]

        })


    except Exception as e:

        print(
            "Failed:",
            feature,
            e
        )


Bio_UniCox = pd.DataFrame(
    bio_uni_results
)

print(
    "Successfully analyzed:",
    len(Bio_UniCox)
)

In [ ]:
# ==========================================================
# FDR correction
# ==========================================================

from statsmodels.stats.multitest import multipletests


Bio_UniCox["FDR"] = multipletests(
    Bio_UniCox["P_value"],
    method="fdr_bh"
)[1]


Bio_UniCox["Direction"] = np.where(
    Bio_UniCox["HR"] > 1,
    "Risk",
    "Protective"
)


Bio_UniCox = (
    Bio_UniCox
    .sort_values(
        "P_value"
    )
    .reset_index(
        drop=True
    )
)


display(
    Bio_UniCox[
        [
            "Feature",
            "Feature_family",
            "HR",
            "P_value",
            "FDR",
            "Direction"
        ]
    ].head(30)
)

In [ ]:
Bio_UniCox.to_csv(
    os.path.join(
        results_dir,
        "Biological_Features_Training_Univariate_Cox.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Significant feature counts by family
# ==========================================================

Bio_UniCox["P005"] = (
    Bio_UniCox["P_value"] < 0.05
)

Bio_UniCox["FDR005"] = (
    Bio_UniCox["FDR"] < 0.05
)


Bio_Significance_Summary = (
    Bio_UniCox
    .groupby(
        "Feature_family"
    )
    .agg(
        Total_features=(
            "Feature",
            "count"
        ),

        P005_features=(
            "P005",
            "sum"
        ),

        FDR005_features=(
            "FDR005",
            "sum"
        )
    )
    .reset_index()
)


display(
    Bio_Significance_Summary
)

In [ ]:
bio_candidates_uni = (
    Bio_UniCox.loc[
        Bio_UniCox[
            "P_value"
        ] < 0.05,
        "Feature"
    ]
    .tolist()
)


print(
    "Biological candidates P<0.05:",
    len(
        bio_candidates_uni
    )
)

print(
    bio_candidates_uni
)

In [ ]:
bio_candidates_fdr = (
    Bio_UniCox.loc[
        Bio_UniCox[
            "FDR"
        ] < 0.05,
        "Feature"
    ]
    .tolist()
)


print(
    "\nBiological candidates FDR<0.05:",
    len(
        bio_candidates_fdr
    )
)

In [ ]:
display(
    Bio_UniCox[
        [
            "Feature",
            "Feature_family",
            "HR",
            "FDR",
            "Direction"
        ]
    ].head(30)
)

In [ ]:
# ==========================================================
# Step15.9.1
# Correlation among biological candidates
# TRAINING SET ONLY
# ==========================================================

bio_corr = (
    train_df[
        bio_candidates_uni
    ]
    .corr(
        method="spearman"
    )
)

print(
    "Biological correlation matrix:",
    bio_corr.shape
)

In [ ]:
# ==========================================================
# Highly correlated biological feature pairs
# ==========================================================

high_corr_bio_pairs = []

for i in range(len(bio_corr.columns)):

    for j in range(i + 1, len(bio_corr.columns)):

        f1 = bio_corr.columns[i]
        f2 = bio_corr.columns[j]

        rho = bio_corr.iloc[i, j]

        if abs(rho) >= 0.80:

            high_corr_bio_pairs.append({

                "Feature1": f1,
                "Feature2": f2,
                "Spearman_rho": rho

            })


High_Corr_Bio_Pairs = pd.DataFrame(
    high_corr_bio_pairs
)


print(
    "Highly correlated pairs |rho| >= 0.80:",
    len(High_Corr_Bio_Pairs)
)

display(
    High_Corr_Bio_Pairs
)

In [ ]:
High_Corr_Bio_Pairs.to_csv(
    os.path.join(
        results_dir,
        "Biological_Features_High_Correlation_Pairs.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step15.9.2
# Biological Elastic Net Cox
# TRAIN SET ONLY
# ==========================================================

from sklearn.preprocessing import StandardScaler
from sksurv.util import Surv


elastic_bio_features = bio_candidates_uni.copy()


X_train_bio = (
    train_df[
        elastic_bio_features
    ]
    .copy()
)


y_train_bio = Surv.from_arrays(
    event=train_df["OS"].astype(bool),
    time=train_df["OS_time"].astype(float)
)


print(
    "Biological X train:",
    X_train_bio.shape
)

In [ ]:
bio_scaler = StandardScaler()

X_train_bio_scaled = bio_scaler.fit_transform(
    X_train_bio
)


X_train_bio_scaled = pd.DataFrame(
    X_train_bio_scaled,
    index=X_train_bio.index,
    columns=X_train_bio.columns
)


print(
    X_train_bio_scaled.shape
)

In [ ]:
# ==========================================================
# Biological Elastic Net alpha path
# ==========================================================

from sksurv.linear_model import CoxnetSurvivalAnalysis
from sklearn.model_selection import KFold
from sksurv.metrics import concordance_index_censored

import numpy as np


bio_coxnet_path = CoxnetSurvivalAnalysis(
    l1_ratio=0.5,
    alpha_min_ratio=0.01,
    n_alphas=100,
    max_iter=100000
)


bio_coxnet_path.fit(
    X_train_bio_scaled,
    y_train_bio
)


bio_alphas = bio_coxnet_path.alphas_


print(
    "Number of alphas:",
    len(bio_alphas)
)

print(
    "Alpha range:",
    bio_alphas.min(),
    "-",
    bio_alphas.max()
)

In [ ]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


bio_alpha_cv_results = []


for alpha in bio_alphas:

    fold_cindices = []

    for train_idx, val_idx in kf.split(
        X_train_bio_scaled
    ):

        X_tr = X_train_bio_scaled.iloc[
            train_idx
        ]

        X_val = X_train_bio_scaled.iloc[
            val_idx
        ]

        y_tr = y_train_bio[
            train_idx
        ]

        y_val = y_train_bio[
            val_idx
        ]


        model = CoxnetSurvivalAnalysis(
            l1_ratio=0.5,
            alphas=[alpha],
            max_iter=100000
        )


        try:

            model.fit(
                X_tr,
                y_tr
            )

            risk_pred = model.predict(
                X_val
            )


            cindex = concordance_index_censored(
                y_val["event"],
                y_val["time"],
                risk_pred
            )[0]


            fold_cindices.append(
                cindex
            )

        except Exception:
            continue


    if len(fold_cindices) > 0:

        bio_alpha_cv_results.append({

            "Alpha": alpha,

            "Mean_Cindex":
                np.mean(
                    fold_cindices
                ),

            "SD_Cindex":
                np.std(
                    fold_cindices
                )

        })


Bio_ElasticNet_CV = pd.DataFrame(
    bio_alpha_cv_results
)


Bio_ElasticNet_CV = (
    Bio_ElasticNet_CV
    .sort_values(
        "Mean_Cindex",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    Bio_ElasticNet_CV.head(10)
)

In [ ]:
best_bio_alpha = (
    Bio_ElasticNet_CV.iloc[0][
        "Alpha"
    ]
)

best_bio_cv_cindex = (
    Bio_ElasticNet_CV.iloc[0][
        "Mean_Cindex"
    ]
)


print(
    "Best biological alpha:",
    best_bio_alpha
)

print(
    "Best biological CV C-index:",
    best_bio_cv_cindex
)

In [ ]:
bio_elastic_model = CoxnetSurvivalAnalysis(
    l1_ratio=0.5,
    alphas=[
        best_bio_alpha
    ],
    max_iter=100000
)


bio_elastic_model.fit(
    X_train_bio_scaled,
    y_train_bio
)

In [ ]:
bio_coef = (
    bio_elastic_model
    .coef_
    .ravel()
)


Bio_ElasticNet_Coefficients = pd.DataFrame({

    "Feature":
        elastic_bio_features,

    "Coefficient":
        bio_coef
})


Bio_ElasticNet_Coefficients[
    "Abs_Coefficient"
] = (
    Bio_ElasticNet_Coefficients[
        "Coefficient"
    ].abs()
)


Bio_ElasticNet_Coefficients[
    "Selected"
] = (
    Bio_ElasticNet_Coefficients[
        "Abs_Coefficient"
    ] > 1e-8
)


Bio_ElasticNet_Coefficients = (
    Bio_ElasticNet_Coefficients
    .sort_values(
        "Abs_Coefficient",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


display(
    Bio_ElasticNet_Coefficients
)

In [ ]:
selected_bio_features = (
    Bio_ElasticNet_Coefficients.loc[
        Bio_ElasticNet_Coefficients[
            "Selected"
        ],
        "Feature"
    ]
    .tolist()
)


print(
    "Selected biological features:",
    len(
        selected_bio_features
    )
)

print(
    selected_bio_features
)

In [ ]:
Bio_ElasticNet_CV.to_csv(
    os.path.join(
        results_dir,
        "Biological_ElasticNet_CV.csv"
    ),
    index=False
)


Bio_ElasticNet_Coefficients.to_csv(
    os.path.join(
        results_dir,
        "Biological_ElasticNet_Coefficients.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step15.10.1
# Random Survival Forest for biological features
# TRAINING SET ONLY
# ==========================================================

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

X_train_bio_rsf = train_df[
    bio_candidates_uni
].copy()

y_train_bio_rsf = Surv.from_arrays(
    event=train_df["OS"].astype(bool),
    time=train_df["OS_time"].astype(float)
)

print("RSF biological X:", X_train_bio_rsf.shape)
print("Biological features:", len(bio_candidates_uni))

In [ ]:
# ==========================================================
# Step15.10.2
# Fit RSF
# ==========================================================

bio_rsf_model = RandomSurvivalForest(
    n_estimators=1000,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

bio_rsf_model.fit(
    X_train_bio_rsf,
    y_train_bio_rsf
)

bio_rsf_train_cindex = bio_rsf_model.score(
    X_train_bio_rsf,
    y_train_bio_rsf
)

print(
    "Biological RSF training C-index:",
    bio_rsf_train_cindex
)

In [ ]:
# ==========================================================
# Step15.10.3
# Permutation importance
# ==========================================================

from sklearn.inspection import permutation_importance
import pandas as pd

bio_perm = permutation_importance(
    bio_rsf_model,
    X_train_bio_rsf,
    y_train_bio_rsf,
    n_repeats=50,
    random_state=42,
    n_jobs=-1
)

Bio_RSF_Importance = pd.DataFrame({
    "Feature": bio_candidates_uni,
    "Importance_mean": bio_perm.importances_mean,
    "Importance_SD": bio_perm.importances_std
})

Bio_RSF_Importance = (
    Bio_RSF_Importance
    .sort_values(
        "Importance_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    Bio_RSF_Importance
)

In [ ]:
Bio_RSF_Importance.to_csv(
    os.path.join(
        results_dir,
        "Biological_RSF_Permutation_Importance.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step15.10.4
# RSF Top10 biological features
# ==========================================================

bio_rsf_top10 = (
    Bio_RSF_Importance
    .head(10)["Feature"]
    .tolist()
)

print(
    "Biological RSF Top10:"
)

print(
    bio_rsf_top10
)

In [ ]:
# ==========================================================
# Step15.10.5
# Integrate Uni-Cox + Elastic Net + RSF
# ==========================================================

Bio_Selection_Integrated = (
    Bio_UniCox[
        [
            "Feature",
            "Feature_family",
            "HR",
            "P_value",
            "FDR",
            "Direction"
        ]
    ]
    .merge(
        Bio_ElasticNet_Coefficients[
            [
                "Feature",
                "Coefficient",
                "Abs_Coefficient",
                "Selected"
            ]
        ],
        on="Feature",
        how="left"
    )
    .merge(
        Bio_RSF_Importance[
            [
                "Feature",
                "Importance_mean",
                "Importance_SD"
            ]
        ],
        on="Feature",
        how="left"
    )
)

In [ ]:
Bio_Selection_Integrated["UniCox_significant"] = (
    Bio_Selection_Integrated["P_value"] < 0.05
)

Bio_Selection_Integrated["ElasticNet_selected"] = (
    Bio_Selection_Integrated["Selected"]
    .fillna(False)
)

Bio_Selection_Integrated["RSF_Top10"] = (
    Bio_Selection_Integrated["Feature"]
    .isin(
        bio_rsf_top10
    )
)

Bio_Selection_Integrated["Selection_Count"] = (
    Bio_Selection_Integrated[
        "UniCox_significant"
    ].astype(int)
    +
    Bio_Selection_Integrated[
        "ElasticNet_selected"
    ].astype(int)
    +
    Bio_Selection_Integrated[
        "RSF_Top10"
    ].astype(int)
)

In [ ]:
Bio_Selection_Integrated = (
    Bio_Selection_Integrated
    .sort_values(
        [
            "Selection_Count",
            "Importance_mean",
            "Abs_Coefficient"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

display(
    Bio_Selection_Integrated[
        [
            "Feature",
            "Feature_family",
            "HR",
            "Direction",
            "FDR",
            "Abs_Coefficient",
            "Importance_mean",
            "UniCox_significant",
            "ElasticNet_selected",
            "RSF_Top10",
            "Selection_Count"
        ]
    ]
)

In [ ]:
# ==========================================================
# Step15.10.6
# Stable biological features
# ==========================================================

stable_bio_features = (
    Bio_Selection_Integrated.loc[
        Bio_Selection_Integrated[
            "Selection_Count"
        ] == 3,
        "Feature"
    ]
    .tolist()
)

print(
    "Stable biological features selected by all 3 methods:",
    len(stable_bio_features)
)

print(
    stable_bio_features
)

In [ ]:
Bio_Selection_Integrated.to_csv(
    os.path.join(
        results_dir,
        "Biological_MultiMethod_Selection_Integrated.csv"
    ),
    index=False
)

pd.DataFrame({
    "Feature": stable_bio_features
}).to_csv(
    os.path.join(
        results_dir,
        "Stable_Biological_Features.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Show stable biological features
# ==========================================================

Stable_Bio_Table = (
    Bio_Selection_Integrated.loc[
        Bio_Selection_Integrated["Selection_Count"] == 3,
        [
            "Feature",
            "Feature_family",
            "HR",
            "Direction",
            "P_value",
            "FDR",
            "Abs_Coefficient",
            "Importance_mean",
            "Selection_Count"
        ]
    ]
    .sort_values(
        "Importance_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

display(Stable_Bio_Table)

print(
    "\nStable biological features:",
    len(Stable_Bio_Table)
)

print(
    Stable_Bio_Table["Feature"].tolist()
)

In [ ]:
stable_bio_features = [
    "mTORC1 Signaling",
    "Checkpoint_TNFRSF4",
    "Cell_B_cells",
    "Function_HLA_activity",
    "Checkpoint_CD276",
    "Pancreas Beta Cells",
    "Spermatogenesis"
]

In [ ]:
clinical_features = [
    "Age",
    "Gender_binary",
    "Stage_numeric",
    "T_numeric"
]

integrated_feature_sets = {

    "Clinical":
        clinical_features,

    "StableGenes10":
        stable_genes,

    "StableBio7":
        stable_bio_features,

    "Genes10_Bio7":
        stable_genes
        + stable_bio_features,

    "Clinical_Genes10":
        clinical_features
        + stable_genes,

    "Clinical_Bio7":
        clinical_features
        + stable_bio_features,

    "Clinical_Genes10_Bio7":
        clinical_features
        + stable_genes
        + stable_bio_features
}

for name, features in integrated_feature_sets.items():
    print(name, len(features))

In [ ]:
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
import pandas as pd

integrated_results = []


for model_name, features in integrated_feature_sets.items():

    train_model = (
        train_df[
            ["OS_time", "OS"] + features
        ]
        .dropna()
        .copy()
    )

    test_model = (
        test_df[
            ["OS_time", "OS"] + features
        ]
        .dropna()
        .copy()
    )


    cph = CoxPHFitter(
        penalizer=0.05
    )

    cph.fit(
        train_model,
        duration_col="OS_time",
        event_col="OS"
    )


    train_risk = (
        cph.predict_partial_hazard(
            train_model[features]
        )
        .values
        .ravel()
    )

    test_risk = (
        cph.predict_partial_hazard(
            test_model[features]
        )
        .values
        .ravel()
    )


    train_cindex = concordance_index(
        train_model["OS_time"],
        -train_risk,
        train_model["OS"]
    )

    test_cindex = concordance_index(
        test_model["OS_time"],
        -test_risk,
        test_model["OS"]
    )


    integrated_results.append({

        "Model": model_name,

        "N_features": len(features),

        "Train_n": len(train_model),

        "Test_n": len(test_model),

        "Train_Cindex":
            train_cindex,

        "Test_Cindex":
            test_cindex,

        "Overfit_gap":
            train_cindex
            -
            test_cindex
    })


Integrated_Cox_Benchmark = pd.DataFrame(
    integrated_results
)

Integrated_Cox_Benchmark = (
    Integrated_Cox_Benchmark
    .sort_values(
        "Test_Cindex",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    Integrated_Cox_Benchmark
)

In [ ]:
Integrated_Cox_Benchmark.to_csv(
    os.path.join(
        results_dir,
        "Integrated_Feature_Sets_CoxPH_Benchmark.csv"
    ),
    index=False
)

In [ ]:
final_candidate_feature_sets = {

    "StableGenes10":
        stable_genes,

    "StableBio7":
        stable_bio_features,

    "Genes10_Bio7":
        stable_genes
        + stable_bio_features
}

for name, features in final_candidate_feature_sets.items():
    print(name, len(features))

In [ ]:
# ==========================================================
# Step15.12
# Multi-algorithm survival model comparison
# TRAIN / TEST
# ==========================================================

import numpy as np
import pandas as pd

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index

from sklearn.preprocessing import StandardScaler

from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import (
    RandomSurvivalForest,
    GradientBoostingSurvivalAnalysis
)

from sksurv.metrics import concordance_index_censored


# ----------------------------------------------------------
# 1. Feature sets
# ----------------------------------------------------------

final_candidate_feature_sets = {

    "StableGenes10":
        stable_genes,

    "StableBio7":
        stable_bio_features,

    "Genes10_Bio7":
        stable_genes + stable_bio_features
}


# ----------------------------------------------------------
# 2. Survival outcome
# ----------------------------------------------------------

y_train = Surv.from_arrays(
    event=train_df["OS"].astype(bool),
    time=train_df["OS_time"].astype(float)
)

y_test = Surv.from_arrays(
    event=test_df["OS"].astype(bool),
    time=test_df["OS_time"].astype(float)
)


all_results = []


# ==========================================================
# Loop over feature sets
# ==========================================================

for feature_set_name, features in final_candidate_feature_sets.items():

    print("\n" + "=" * 70)
    print(feature_set_name, len(features))
    print("=" * 70)

    X_train = train_df[features].copy()
    X_test = test_df[features].copy()


    # ======================================================
    # A. CoxPH
    # ======================================================

    train_cox = train_df[
        ["OS_time", "OS"] + features
    ].copy()

    test_cox = test_df[
        ["OS_time", "OS"] + features
    ].copy()

    cph = CoxPHFitter(
        penalizer=0.05
    )

    cph.fit(
        train_cox,
        duration_col="OS_time",
        event_col="OS"
    )

    train_risk = (
        cph.predict_partial_hazard(
            X_train
        )
        .values
        .ravel()
    )

    test_risk = (
        cph.predict_partial_hazard(
            X_test
        )
        .values
        .ravel()
    )

    train_ci = concordance_index(
        train_df["OS_time"],
        -train_risk,
        train_df["OS"]
    )

    test_ci = concordance_index(
        test_df["OS_time"],
        -test_risk,
        test_df["OS"]
    )

    all_results.append({
        "Feature_Set": feature_set_name,
        "Algorithm": "CoxPH",
        "N_features": len(features),
        "Train_Cindex": train_ci,
        "Test_Cindex": test_ci,
        "Overfit_gap": train_ci - test_ci
    })


    # ======================================================
    # B. Elastic Net Cox
    # ======================================================

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    )

    X_test_scaled = scaler.transform(
        X_test
    )


    # alpha path
    coxnet_path = CoxnetSurvivalAnalysis(
        l1_ratio=0.5,
        alpha_min_ratio=0.01,
        n_alphas=100,
        max_iter=100000
    )

    coxnet_path.fit(
        X_train_scaled,
        y_train
    )

    # 简化：用前面路径中中等偏小惩罚
    # 后面如需要再做严格CV优化
    alpha_index = int(
        len(coxnet_path.alphas_) * 0.65
    )

    chosen_alpha = (
        coxnet_path.alphas_[alpha_index]
    )

    coxnet = CoxnetSurvivalAnalysis(
        l1_ratio=0.5,
        alphas=[chosen_alpha],
        max_iter=100000
    )

    coxnet.fit(
        X_train_scaled,
        y_train
    )

    train_risk = coxnet.predict(
        X_train_scaled
    )

    test_risk = coxnet.predict(
        X_test_scaled
    )

    train_ci = concordance_index_censored(
        y_train["event"],
        y_train["time"],
        train_risk
    )[0]

    test_ci = concordance_index_censored(
        y_test["event"],
        y_test["time"],
        test_risk
    )[0]

    all_results.append({
        "Feature_Set": feature_set_name,
        "Algorithm": "ElasticNet_Cox",
        "N_features": len(features),
        "Train_Cindex": train_ci,
        "Test_Cindex": test_ci,
        "Overfit_gap": train_ci - test_ci
    })


    # ======================================================
    # C. Random Survival Forest
    # ======================================================

    rsf = RandomSurvivalForest(
        n_estimators=1000,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    )

    rsf.fit(
        X_train,
        y_train
    )

    train_ci = rsf.score(
        X_train,
        y_train
    )

    test_ci = rsf.score(
        X_test,
        y_test
    )

    all_results.append({
        "Feature_Set": feature_set_name,
        "Algorithm": "RSF",
        "N_features": len(features),
        "Train_Cindex": train_ci,
        "Test_Cindex": test_ci,
        "Overfit_gap": train_ci - test_ci
    })


    # ======================================================
    # D. Gradient Boosting Survival
    # ======================================================

    gbs = GradientBoostingSurvivalAnalysis(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=2,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    )

    gbs.fit(
        X_train,
        y_train
    )

    train_risk = gbs.predict(
        X_train
    )

    test_risk = gbs.predict(
        X_test
    )

    train_ci = concordance_index_censored(
        y_train["event"],
        y_train["time"],
        train_risk
    )[0]

    test_ci = concordance_index_censored(
        y_test["event"],
        y_test["time"],
        test_risk
    )[0]

    all_results.append({
        "Feature_Set": feature_set_name,
        "Algorithm": "GBM_Survival",
        "N_features": len(features),
        "Train_Cindex": train_ci,
        "Test_Cindex": test_ci,
        "Overfit_gap": train_ci - test_ci
    })


# ==========================================================
# Final result table
# ==========================================================

MultiAlgo_Results = pd.DataFrame(
    all_results
)

MultiAlgo_Results = (
    MultiAlgo_Results
    .sort_values(
        "Test_Cindex",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    MultiAlgo_Results
)

In [ ]:
MultiAlgo_Results.to_csv(
    os.path.join(
        results_dir,
        "MultiAlgorithm_Survival_Model_Comparison.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step15.13
# Time-dependent AUC at 1 / 2 / 3 years
# Genes10_Bio7
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

from sksurv.util import Surv
from sksurv.metrics import cumulative_dynamic_auc
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest

from lifelines import CoxPHFitter


# ----------------------------------------------------------
# Features
# ----------------------------------------------------------

final_features = (
    stable_genes
    + stable_bio_features
)

X_train = train_df[
    final_features
].copy()

X_test = test_df[
    final_features
].copy()


y_train = Surv.from_arrays(
    event=train_df["OS"].astype(bool),
    time=train_df["OS_time"].astype(float)
)

y_test = Surv.from_arrays(
    event=test_df["OS"].astype(bool),
    time=test_df["OS_time"].astype(float)
)


# ----------------------------------------------------------
# Evaluation times
# IMPORTANT:
# if OS_time is in days, use 365/730/1095
# ----------------------------------------------------------

times = np.array([
    365,
    730,
    1095
], dtype=float)


print("Evaluation times:", times)

In [ ]:
print(
    train_df["OS_time"].describe()
)

In [ ]:
# ==========================================================
# CoxPH
# ==========================================================

cox_train_df = train_df[
    ["OS_time", "OS"] + final_features
].copy()

cox_model = CoxPHFitter(
    penalizer=0.05
)

cox_model.fit(
    cox_train_df,
    duration_col="OS_time",
    event_col="OS"
)

cox_test_risk = (
    cox_model
    .predict_partial_hazard(
        X_test
    )
    .values
    .ravel()
)


cox_auc, cox_mean_auc = cumulative_dynamic_auc(
    y_train,
    y_test,
    cox_test_risk,
    times
)

print(
    "CoxPH AUC:",
    cox_auc
)

In [ ]:
# ==========================================================
# Elastic Net Cox
# ==========================================================

scaler_final = StandardScaler()

X_train_scaled = scaler_final.fit_transform(
    X_train
)

X_test_scaled = scaler_final.transform(
    X_test
)


elastic_final = CoxnetSurvivalAnalysis(
    l1_ratio=0.5,
    alphas=[chosen_alpha],
    max_iter=100000
)

elastic_final.fit(
    X_train_scaled,
    y_train
)

elastic_test_risk = elastic_final.predict(
    X_test_scaled
)


elastic_auc, elastic_mean_auc = cumulative_dynamic_auc(
    y_train,
    y_test,
    elastic_test_risk,
    times
)

print(
    "ElasticNet AUC:",
    elastic_auc
)

In [ ]:
# ==========================================================
# Random Survival Forest
# ==========================================================

rsf_final = RandomSurvivalForest(
    n_estimators=1000,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rsf_final.fit(
    X_train,
    y_train
)

rsf_test_risk = rsf_final.predict(
    X_test
)


rsf_auc, rsf_mean_auc = cumulative_dynamic_auc(
    y_train,
    y_test,
    rsf_test_risk,
    times
)

print(
    "RSF AUC:",
    rsf_auc
)

In [ ]:
# ==========================================================
# Summary
# ==========================================================

TimeAUC_Results = pd.DataFrame({

    "Model": [
        "CoxPH",
        "ElasticNet_Cox",
        "RSF"
    ],

    "AUC_1yr": [
        cox_auc[0],
        elastic_auc[0],
        rsf_auc[0]
    ],

    "AUC_2yr": [
        cox_auc[1],
        elastic_auc[1],
        rsf_auc[1]
    ],

    "AUC_3yr": [
        cox_auc[2],
        elastic_auc[2],
        rsf_auc[2]
    ]
})


display(
    TimeAUC_Results
)

In [ ]:
TimeAUC_Results.to_csv(
    os.path.join(
        results_dir,
        "Genes10_Bio7_TimeDependent_AUC.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Output directory
# ==========================================================

import os

fig_dir = r".\LIHC_RNA_Step15_IntegratedModel\Figures"
os.makedirs(fig_dir, exist_ok=True)

In [ ]:
# ==========================================================
# Visualization 1
# Time-dependent AUC line plot
# ==========================================================

import matplotlib.pyplot as plt
import numpy as np

plot_df = TimeAUC_Results.copy()

time_labels = ["1-year", "2-year", "3-year"]
x = np.arange(len(time_labels))

plt.figure(figsize=(7,5))

for _, row in plot_df.iterrows():
    plt.plot(
        x,
        [row["AUC_1yr"], row["AUC_2yr"], row["AUC_3yr"]],
        marker="o",
        linewidth=2,
        markersize=7,
        label=row["Model"]
    )

plt.xticks(x, time_labels, fontsize=11)
plt.yticks(fontsize=11)
plt.ylim(0.75, 0.90)

plt.xlabel("Time point", fontsize=12)
plt.ylabel("Time-dependent AUC", fontsize=12)
plt.title("Time-dependent AUC of Survival Models", fontsize=14)

plt.legend(frameon=False)
plt.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()

plt.savefig(
    os.path.join(fig_dir, "TimeDependent_AUC_lineplot.pdf"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(fig_dir, "TimeDependent_AUC_lineplot.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================================
# Visualization 2
# Grouped bar plot
# ==========================================================

import matplotlib.pyplot as plt
import numpy as np

plot_df = TimeAUC_Results.copy()

models = plot_df["Model"].tolist()
x = np.arange(len(models))
width = 0.22

auc_1 = plot_df["AUC_1yr"].values
auc_2 = plot_df["AUC_2yr"].values
auc_3 = plot_df["AUC_3yr"].values

plt.figure(figsize=(8,5))

bars1 = plt.bar(x - width, auc_1, width=width, label="1-year")
bars2 = plt.bar(x,         auc_2, width=width, label="2-year")
bars3 = plt.bar(x + width, auc_3, width=width, label="3-year")

plt.xticks(x, models, fontsize=11)
plt.yticks(fontsize=11)
plt.ylim(0.75, 0.90)

plt.xlabel("Model", fontsize=12)
plt.ylabel("Time-dependent AUC", fontsize=12)
plt.title("Comparison of Time-dependent AUC Across Models", fontsize=14)

plt.legend(frameon=False)
plt.grid(axis="y", linestyle="--", alpha=0.4)

# 数值标注
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2,
            h + 0.002,
            f"{h:.3f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()

plt.savefig(
    os.path.join(fig_dir, "TimeDependent_AUC_barplot.pdf"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(fig_dir, "TimeDependent_AUC_barplot.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================================
# Visualization 3
# Publication-style simple line plot
# ==========================================================

import matplotlib.pyplot as plt
import numpy as np

plot_df = TimeAUC_Results.copy()

time_labels = ["1-year", "2-year", "3-year"]
x = np.arange(1, 4)

plt.figure(figsize=(6.5,4.8))

for _, row in plot_df.iterrows():
    y = [row["AUC_1yr"], row["AUC_2yr"], row["AUC_3yr"]]
    plt.plot(
        x,
        y,
        marker="o",
        linewidth=2,
        markersize=6,
        label=row["Model"]
    )
    
    for xi, yi in zip(x, y):
        plt.text(
            xi,
            yi + 0.003,
            f"{yi:.3f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.xticks(x, time_labels, fontsize=11)
plt.yticks(fontsize=11)
plt.ylim(0.80, 0.89)

plt.xlabel("Time point", fontsize=12)
plt.ylabel("AUC", fontsize=12)
plt.title("Time-dependent ROC Performance", fontsize=14)

plt.legend(frameon=False, loc="lower left")
plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(fig_dir, "TimeDependent_AUC_lineplot_clean.pdf"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(fig_dir, "TimeDependent_AUC_lineplot_clean.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ==========================================================
# Step15 Visualization A
# Test C-index comparison
# ==========================================================

import os
import matplotlib.pyplot as plt
import pandas as pd

fig_dir = os.path.join(
    _hcc_path('LIHC_RNA_Step15_Machine_Learning'),
    "Figures"
)

os.makedirs(fig_dir, exist_ok=True)


# ----------------------------------------------------------
# Genes10_Bio7 models only
# ----------------------------------------------------------

plot_cindex = (
    MultiAlgo_Results.loc[
        MultiAlgo_Results["Feature_Set"] == "Genes10_Bio7",
        [
            "Algorithm",
            "Train_Cindex",
            "Test_Cindex"
        ]
    ]
    .copy()
)


plot_cindex = plot_cindex.sort_values(
    "Test_Cindex",
    ascending=False
)


display(plot_cindex)

In [ ]:
# ==========================================================
# C-index grouped bar plot
# ==========================================================

import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(
    figsize=(8, 5.5)
)

x = np.arange(
    len(plot_cindex)
)

width = 0.34


bars_train = ax.bar(
    x - width / 2,
    plot_cindex["Train_Cindex"],
    width,
    label="Training"
)

bars_test = ax.bar(
    x + width / 2,
    plot_cindex["Test_Cindex"],
    width,
    label="Testing"
)


ax.set_xticks(x)

ax.set_xticklabels(
    plot_cindex["Algorithm"],
    rotation=15,
    ha="right"
)

ax.set_ylabel(
    "C-index",
    fontsize=12
)

ax.set_xlabel(
    "Survival algorithm",
    fontsize=12
)

ax.set_title(
    "Concordance Performance of Integrated Survival Models",
    fontsize=13
)

ax.set_ylim(
    0.65,
    0.95
)


# ----------------------------------------------------------
# Add values
# ----------------------------------------------------------

for bars in [
    bars_train,
    bars_test
]:

    for bar in bars:

        value = bar.get_height()

        ax.text(
            bar.get_x()
            + bar.get_width() / 2,
            value + 0.006,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=9
        )


ax.legend(
    frameon=False
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

plt.tight_layout()


# ----------------------------------------------------------
# SAVE BEFORE SHOW
# ----------------------------------------------------------

fig.savefig(
    os.path.join(
        fig_dir,
        "Fig15A_MultiAlgorithm_Cindex.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "Fig15A_MultiAlgorithm_Cindex.svg"
    ),
    bbox_inches="tight"
)

plt.show()

plt.close(fig)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ==========================================================
# Step15.14
# Rebuild 3 final Genes10_Bio7 survival models
# CoxPH + ElasticNet Cox + RSF
# ==========================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import CoxPHFitter
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

from sklearn.preprocessing import StandardScaler

from sksurv.util import Surv
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest


# ==========================================================
# Paths
# ==========================================================

root_dir = _hcc_path('LIHC_RNA_Step15_Machine_Learning')

results_dir = os.path.join(
    root_dir,
    "Results"
)

fig_dir = os.path.join(
    root_dir,
    "Figures"
)

os.makedirs(results_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)


# ==========================================================
# Final 17 features
# ==========================================================

final_features = (
    stable_genes
    +
    stable_bio_features
)

print("Final feature number:", len(final_features))
print(final_features)


# ==========================================================
# Survival outcomes
# ==========================================================

y_train = Surv.from_arrays(
    event=train_df["OS"].astype(bool),
    time=train_df["OS_time"].astype(float)
)

y_test = Surv.from_arrays(
    event=test_df["OS"].astype(bool),
    time=test_df["OS_time"].astype(float)
)


X_train = train_df[
    final_features
].copy()

X_test = test_df[
    final_features
].copy()

In [ ]:
# ==========================================================
# CoxPH
# ==========================================================

train_cox_df = train_df[
    ["OS_time", "OS"] + final_features
].copy()


final_cox = CoxPHFitter(
    penalizer=0.05
)

final_cox.fit(
    train_cox_df,
    duration_col="OS_time",
    event_col="OS"
)


cox_train_risk = (
    final_cox
    .predict_partial_hazard(
        X_train
    )
    .values
    .ravel()
)


cox_test_risk = (
    final_cox
    .predict_partial_hazard(
        X_test
    )
    .values
    .ravel()
)

In [ ]:
# ==========================================================
# ElasticNet Cox
# ==========================================================

final_scaler = StandardScaler()

X_train_scaled = final_scaler.fit_transform(
    X_train
)

X_test_scaled = final_scaler.transform(
    X_test
)


final_elastic = CoxnetSurvivalAnalysis(
    l1_ratio=0.5,
    alphas=[chosen_alpha],
    max_iter=100000,

    # 为后面的 calibration 计算生存概率
    fit_baseline_model=True
)


final_elastic.fit(
    X_train_scaled,
    y_train
)


elastic_train_risk = final_elastic.predict(
    X_train_scaled
)

elastic_test_risk = final_elastic.predict(
    X_test_scaled
)

In [ ]:
# ==========================================================
# Random Survival Forest
# ==========================================================

final_rsf = RandomSurvivalForest(
    n_estimators=1000,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)


final_rsf.fit(
    X_train,
    y_train
)


rsf_train_risk = final_rsf.predict(
    X_train
)

rsf_test_risk = final_rsf.predict(
    X_test
)

In [ ]:
# ==========================================================
# Risk score table
# ==========================================================

train_risk_df = train_df[
    ["OS_time", "OS"]
].copy()

test_risk_df = test_df[
    ["OS_time", "OS"]
].copy()


train_risk_df["CoxPH"] = cox_train_risk
test_risk_df["CoxPH"] = cox_test_risk

train_risk_df["ElasticNet_Cox"] = elastic_train_risk
test_risk_df["ElasticNet_Cox"] = elastic_test_risk

train_risk_df["RSF"] = rsf_train_risk
test_risk_df["RSF"] = rsf_test_risk

In [ ]:
# ==========================================================
# Training-derived median cutoffs
# ==========================================================

risk_cutoffs = {}

for model in [
    "CoxPH",
    "ElasticNet_Cox",
    "RSF"
]:

    risk_cutoffs[model] = np.median(
        train_risk_df[model]
    )

    print(
        model,
        "cutoff =",
        risk_cutoffs[model]
    )

In [ ]:
# ==========================================================
# KM plotting function
# ==========================================================

def plot_test_km(
    model_name,
    train_risk_data,
    test_risk_data,
    cutoff,
    fig_dir
):

    temp = test_risk_data[
        ["OS_time", "OS", model_name]
    ].copy()


    temp["RiskGroup"] = np.where(
        temp[model_name] >= cutoff,
        "High risk",
        "Low risk"
    )


    print("\n", model_name)

    print(
        temp["RiskGroup"]
        .value_counts()
    )


    high = (
        temp["RiskGroup"]
        == "High risk"
    )

    low = (
        temp["RiskGroup"]
        == "Low risk"
    )


    # ------------------------------------------------------
    # Log-rank
    # ------------------------------------------------------

    lr = logrank_test(

        temp.loc[
            high,
            "OS_time"
        ],

        temp.loc[
            low,
            "OS_time"
        ],

        event_observed_A=temp.loc[
            high,
            "OS"
        ],

        event_observed_B=temp.loc[
            low,
            "OS"
        ]
    )


    # ------------------------------------------------------
    # Plot
    # ------------------------------------------------------

    kmf = KaplanMeierFitter()

    fig, ax = plt.subplots(
        figsize=(7, 6)
    )


    for group in [
        "Low risk",
        "High risk"
    ]:

        mask = (
            temp["RiskGroup"]
            == group
        )

        kmf.fit(

            temp.loc[
                mask,
                "OS_time"
            ],

            event_observed=temp.loc[
                mask,
                "OS"
            ],

            label=group
        )


        kmf.plot_survival_function(
            ax=ax,
            ci_show=True,
            linewidth=2
        )


    ax.set_xlabel(
        "Overall survival time (days)",
        fontsize=12
    )

    ax.set_ylabel(
        "Overall survival probability",
        fontsize=12
    )


    ax.set_title(
        f"{model_name}: Testing Cohort",
        fontsize=14
    )


    ax.text(
        0.05,
        0.08,
        f"Log-rank P = {lr.p_value:.3e}",
        transform=ax.transAxes,
        fontsize=11
    )


    ax.grid(
        linestyle="--",
        alpha=0.25
    )

    ax.legend(
        frameon=False
    )


    plt.tight_layout()


    # ------------------------------------------------------
    # SAVE BEFORE SHOW
    # ------------------------------------------------------

    safe_name = (
        model_name
        .replace(" ", "_")
    )


    fig.savefig(
        os.path.join(
            fig_dir,
            f"Fig15_KM_{safe_name}_Test.png"
        ),
        dpi=600,
        bbox_inches="tight"
    )


    fig.savefig(
        os.path.join(
            fig_dir,
            f"Fig15_KM_{safe_name}_Test.svg"
        ),
        bbox_inches="tight"
    )


    plt.show()

    plt.close(fig)


    return temp, lr.p_value

In [ ]:
# ==========================================================
# Run KM for all 3 models
# ==========================================================

km_results = []


for model in [
    "CoxPH",
    "ElasticNet_Cox",
    "RSF"
]:

    temp, p = plot_test_km(

        model_name=model,

        train_risk_data=train_risk_df,

        test_risk_data=test_risk_df,

        cutoff=risk_cutoffs[model],

        fig_dir=fig_dir
    )


    km_results.append({

        "Model":
            model,

        "Cutoff":
            risk_cutoffs[model],

        "High_n":
            (temp["RiskGroup"] == "High risk").sum(),

        "Low_n":
            (temp["RiskGroup"] == "Low risk").sum(),

        "LogRank_P":
            p
    })


KM_Results = pd.DataFrame(
    km_results
)


display(KM_Results)

In [ ]:
KM_Results.to_csv(
    os.path.join(
        results_dir,
        "Final3Models_Test_KM_Results.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step15.15
# Predicted survival probabilities
# ==========================================================

evaluation_times = np.array(
    [
        365,
        730,
        1095
    ],
    dtype=float
)

In [ ]:
cox_surv = (
    final_cox
    .predict_survival_function(
        X_test,
        times=evaluation_times
    )
    .T
    .values
)

print(
    "Cox survival:",
    cox_surv.shape
)

In [ ]:
elastic_surv_functions = (
    final_elastic
    .predict_survival_function(
        X_test_scaled
    )
)


elastic_surv = np.array([

    [
        fn(t)
        for t in evaluation_times
    ]

    for fn in elastic_surv_functions
])


print(
    "ElasticNet survival:",
    elastic_surv.shape
)

In [ ]:
rsf_surv_functions = (
    final_rsf
    .predict_survival_function(
        X_test
    )
)


rsf_surv = np.array([

    [
        fn(t)
        for t in evaluation_times
    ]

    for fn in rsf_surv_functions
])


print(
    "RSF survival:",
    rsf_surv.shape
)

In [ ]:
survival_predictions = {

    "CoxPH":
        cox_surv,

    "ElasticNet_Cox":
        elastic_surv,

    "RSF":
        rsf_surv
}

In [ ]:
# ==========================================================
# Calibration function
# ==========================================================

def calculate_survival_calibration(
    os_time,
    os_event,
    pred_surv,
    time_point,
    n_bins=5
):

    temp = pd.DataFrame({

        "OS_time":
            np.asarray(os_time),

        "OS":
            np.asarray(os_event),

        "Predicted":
            np.asarray(pred_surv)
    })


    temp["Bin"] = pd.qcut(
        temp["Predicted"],
        q=n_bins,
        duplicates="drop"
    )


    result = []


    for _, group in temp.groupby(
        "Bin",
        observed=True
    ):

        kmf = KaplanMeierFitter()

        kmf.fit(
            group["OS_time"],
            event_observed=group["OS"]
        )


        observed_survival = kmf.predict(
            time_point
        )


        result.append({

            "Predicted":
                group["Predicted"].mean(),

            "Observed":
                observed_survival,

            "N":
                len(group)
        })


    return pd.DataFrame(
        result
    )

In [ ]:
# ==========================================================
# Calibration plots
# 3 models × 3 time points
# ==========================================================

time_labels = [
    "1-year",
    "2-year",
    "3-year"
]


all_calibration_results = []


for model_name, surv_matrix in survival_predictions.items():

    for j, (
        time_point,
        time_label
    ) in enumerate(
        zip(
            evaluation_times,
            time_labels
        )
    ):


        cal = calculate_survival_calibration(

            os_time=test_df["OS_time"],

            os_event=test_df["OS"],

            pred_surv=surv_matrix[:, j],

            time_point=time_point,

            n_bins=5
        )


        cal["Model"] = model_name
        cal["Time"] = time_label


        all_calibration_results.append(
            cal
        )


        # --------------------------------------------------
        # Plot
        # --------------------------------------------------

        fig, ax = plt.subplots(
            figsize=(5.8, 5.5)
        )


        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            linewidth=1.5,
            label="Ideal"
        )


        ax.plot(
            cal["Predicted"],
            cal["Observed"],
            marker="o",
            markersize=7,
            linewidth=2,
            label=model_name
        )


        ax.set_xlim(
            0,
            1
        )

        ax.set_ylim(
            0,
            1
        )


        ax.set_xlabel(
            "Predicted survival probability",
            fontsize=11
        )

        ax.set_ylabel(
            "Observed survival probability",
            fontsize=11
        )


        ax.set_title(
            f"{model_name}: {time_label} Calibration",
            fontsize=13
        )


        ax.grid(
            linestyle="--",
            alpha=0.25
        )

        ax.legend(
            frameon=False
        )


        plt.tight_layout()


        safe_model = model_name.replace(
            " ",
            "_"
        )

        safe_time = time_label.replace(
            "-",
            "_"
        )


        # ----------------------------------------------
        # SAVE BEFORE SHOW
        # ----------------------------------------------

        fig.savefig(
            os.path.join(
                fig_dir,
                f"Fig15_Calibration_{safe_model}_{safe_time}.png"
            ),
            dpi=600,
            bbox_inches="tight"
        )


        fig.savefig(
            os.path.join(
                fig_dir,
                f"Fig15_Calibration_{safe_model}_{safe_time}.svg"
            ),
            bbox_inches="tight"
        )


        plt.show()

        plt.close(fig)

In [ ]:
Calibration_All = pd.concat(
    all_calibration_results,
    ignore_index=True
)


Calibration_All.to_csv(
    os.path.join(
        results_dir,
        "Final3Models_Calibration_Data.csv"
    ),
    index=False
)

In [ ]:
# ==========================================================
# Step15.16
# DCA helper functions
# ==========================================================

def build_landmark_dca_data(
    os_time,
    os_event,
    predicted_survival,
    time_point
):

    temp = pd.DataFrame({

        "OS_time":
            np.asarray(os_time),

        "OS":
            np.asarray(os_event),

        "PredSurv":
            np.asarray(predicted_survival)
    })


    # ------------------------------------------------------
    # Known outcome at target time
    # ------------------------------------------------------

    known = (

        (
            (temp["OS"] == 1)
            &
            (temp["OS_time"] <= time_point)
        )

        |

        (
            temp["OS_time"] > time_point
        )
    )


    temp = temp.loc[
        known
    ].copy()


    temp["Event_by_time"] = (

        (
            temp["OS"] == 1
        )

        &

        (
            temp["OS_time"] <= time_point
        )

    ).astype(int)


    temp["Predicted_event"] = (
        1
        -
        temp["PredSurv"]
    )


    return temp

In [ ]:
def net_benefit_curve(
    y_true,
    pred_prob,
    thresholds
):

    y_true = np.asarray(
        y_true
    )

    pred_prob = np.asarray(
        pred_prob
    )


    n = len(
        y_true
    )


    net_benefit = []


    for pt in thresholds:

        positive = (
            pred_prob >= pt
        )


        tp = np.sum(
            positive
            &
            (y_true == 1)
        )


        fp = np.sum(
            positive
            &
            (y_true == 0)
        )


        nb = (

            tp / n

            -

            fp / n

            *
            pt / (1 - pt)
        )


        net_benefit.append(
            nb
        )


    return np.asarray(
        net_benefit
    )

In [ ]:
# ==========================================================
# DCA
# Compare all 3 models on same figure
# ==========================================================

thresholds = np.arange(
    0.05,
    0.81,
    0.01
)


DCA_results = []


for j, (
    time_point,
    time_label
) in enumerate(
    zip(
        evaluation_times,
        time_labels
    )
):


    fig, ax = plt.subplots(
        figsize=(7.2, 5.8)
    )


    reference_dca = None


    # ======================================================
    # Three models
    # ======================================================

    for model_name, surv_matrix in survival_predictions.items():


        dca_df = build_landmark_dca_data(

            os_time=test_df["OS_time"],

            os_event=test_df["OS"],

            predicted_survival=surv_matrix[:, j],

            time_point=time_point
        )


        model_nb = net_benefit_curve(

            y_true=dca_df[
                "Event_by_time"
            ],

            pred_prob=dca_df[
                "Predicted_event"
            ],

            thresholds=thresholds
        )


        ax.plot(
            thresholds,
            model_nb,
            linewidth=2,
            label=model_name
        )


        for threshold, nb in zip(
            thresholds,
            model_nb
        ):

            DCA_results.append({

                "Model":
                    model_name,

                "Time":
                    time_label,

                "Threshold":
                    threshold,

                "NetBenefit":
                    nb,

                "N_evaluable":
                    len(dca_df)
            })


        reference_dca = dca_df


    # ======================================================
    # Treat all / treat none
    # ======================================================

    event_rate = (
        reference_dca[
            "Event_by_time"
        ].mean()
    )


    treat_all = (

        event_rate

        -

        (
            1 - event_rate
        )

        *
        thresholds

        /
        (
            1 - thresholds
        )
    )


    treat_none = np.zeros_like(
        thresholds
    )


    ax.plot(
        thresholds,
        treat_all,
        linestyle="--",
        linewidth=1.5,
        label="Treat all"
    )


    ax.plot(
        thresholds,
        treat_none,
        linestyle=":",
        linewidth=1.5,
        label="Treat none"
    )


    # ======================================================
    # Figure settings
    # ======================================================

    ax.set_xlabel(
        "Threshold probability",
        fontsize=12
    )

    ax.set_ylabel(
        "Net benefit",
        fontsize=12
    )


    ax.set_title(
        f"{time_label} Decision Curve Analysis",
        fontsize=13
    )


    ax.set_xlim(
        0.05,
        0.80
    )


    ax.set_ylim(
        -0.05,
        0.35
    )


    ax.grid(
        linestyle="--",
        alpha=0.25
    )


    ax.legend(
        frameon=False
    )


    plt.tight_layout()


    safe_time = time_label.replace(
        "-",
        "_"
    )


    # ======================================================
    # SAVE BEFORE SHOW
    # ======================================================

    fig.savefig(
        os.path.join(
            fig_dir,
            f"Fig15_DCA_3Models_{safe_time}.png"
        ),
        dpi=600,
        bbox_inches="tight"
    )


    fig.savefig(
        os.path.join(
            fig_dir,
            f"Fig15_DCA_3Models_{safe_time}.svg"
        ),
        bbox_inches="tight"
    )


    plt.show()

    plt.close(fig)

In [ ]:
DCA_All = pd.DataFrame(
    DCA_results
)


DCA_All.to_csv(
    os.path.join(
        results_dir,
        "Final3Models_DCA_Data.csv"
    ),
    index=False
)